# 03 - Querying with Prompting

In [ ]:
%reload_ext dotenv
%dotenv

import os

from graphrag_toolkit.lexical_graph.storage.graph.neo4j_graph_store_factory import Neo4jGraphStoreFactory
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.lexical_graph.storage import VectorStoreFactory
from graphrag_toolkit.lexical_graph import set_logging_config

set_logging_config('INFO')

# Register the Neo4j backend with the factory
GraphStoreFactory.register(Neo4jGraphStoreFactory)

# Create graph and vector stores
graph_store = GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE'])
vector_store = VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE'])

### TraversalBasedRetriever

See [TraversalBasedRetriever](https://github.com/awslabs/graphrag-toolkit/blob/main/docs/lexical-graph/querying.md#traversalbasedretriever).

In [ ]:
from graphrag_toolkit.lexical_graph import LexicalGraphQueryEngine

query_engine = LexicalGraphQueryEngine.for_traversal_based_search(
    graph_store, 
    vector_store,
    streaming=True
)

response = query_engine.query("What are the similarities and differences between Neptune Database and Neptune Analytics?")

print(response.print_response_stream())

## Set prompt from disk

In [ ]:
from graphrag_toolkit.lexical_graph import LexicalGraphQueryEngine
from graphrag_toolkit.lexical_graph.prompts.file_prompt_provider import FilePromptProvider
from graphrag_toolkit.lexical_graph.prompts.prompt_provider_config import FilePromptProviderConfig

# Setup prompt provider
prompt_provider = FilePromptProvider(
    FilePromptProviderConfig(
        system_prompt_file="prompts/system_prompt.txt",
        user_prompt_file="prompts/user_prompt.txt"
    )
)

# Create query engine with prompt provider
query_engine = LexicalGraphQueryEngine.for_traversal_based_search(
    graph_store, 
    vector_store,
    streaming=True,
    prompt_provider=prompt_provider
)

response = query_engine.query("What are the similarities and differences between Neptune Database and Neptune Analytics?")

print(response.print_response_stream())


## Set prompt from S3

In [ ]:
import os

from graphrag_toolkit.lexical_graph import LexicalGraphQueryEngine
from graphrag_toolkit.lexical_graph.prompts.s3_prompt_provider import S3PromptProvider
from graphrag_toolkit.lexical_graph.prompts.prompt_provider_config import S3PromptProviderConfig

# Setup S3 prompt provider
s3_bucket = os.environ.get('S3_BUCKET_NAME', '')

if not s3_bucket:
    print('⚠️  S3_BUCKET_NAME not set in .env — skipping S3 prompt provider demo.')
    print('Set S3_BUCKET_NAME in notebooks/.env and upload prompt files to use this feature.')
else:
    prompt_provider = S3PromptProvider(
        S3PromptProviderConfig(
            bucket=s3_bucket,
            prefix=os.environ.get('PROMPT_PREFIX', 'prompts'),
            system_prompt_file="system_prompt.txt",
            user_prompt_file="user_prompt.txt"
        )
    )

    # Create query engine with S3 prompts
    query_engine = LexicalGraphQueryEngine.for_traversal_based_search(
        graph_store, 
        vector_store,
        streaming=True,
        prompt_provider=prompt_provider
    )

    response = query_engine.query("What are the similarities and differences between Neptune Database and Neptune Analytics?")

    print(response.print_response_stream())


## Bedrock Manage Prompt

In [ ]:
from graphrag_toolkit.lexical_graph import LexicalGraphQueryEngine
from graphrag_toolkit.lexical_graph.prompts.bedrock_prompt_provider import BedrockPromptProvider
from graphrag_toolkit.lexical_graph.prompts.prompt_provider_config import BedrockPromptProviderConfig

# Setup Bedrock prompt provider
# Requires SYSTEM_PROMPT_ARN and USER_PROMPT_ARN in .env
system_prompt_arn = os.environ.get('SYSTEM_PROMPT_ARN', '')
user_prompt_arn = os.environ.get('USER_PROMPT_ARN', '')

if not system_prompt_arn or not user_prompt_arn:
    print('⚠️  SYSTEM_PROMPT_ARN and/or USER_PROMPT_ARN not set in .env — skipping Bedrock prompt provider demo.')
    print('Run the aws/create_custom_prompt.sh script to create managed prompts, then add the ARNs to .env.')
else:
    prompt_provider = BedrockPromptProvider(
        BedrockPromptProviderConfig()
    )

    # Create query engine with Bedrock prompts
    query_engine = LexicalGraphQueryEngine.for_traversal_based_search(
        graph_store, 
        vector_store,
        streaming=True,
        prompt_provider=prompt_provider
    )

    response = query_engine.query("What are the similarities and differences between Neptune Database and Neptune Analytics?")

    print(response.print_response_stream())
